# 다중 세션 평가

이 노트북에서는 LLM을 판정자로 사용하는 확장 가능한 LLM 기반 평가 프레임워크인 Strands Evals로 에이전트 세션을 평가합니다. 각 세션에 대해 AgentCore Observability에서 trace를 가져와 evaluator를 실행하고, 대시보드에서 연계해 볼 수 있도록 원본 trace ID와 함께 결과를 다시 기록합니다.

**이 노트북에서 다루는 두 가지 evaluator:**
- **OutputEvaluator**: 응답 품질(관련성, 정확성, 완전성)을 평가합니다.
- **TrajectoryEvaluator**: tool 사용(선택, 효율성, 순서)을 평가합니다.

Strands Evals는 거의 모든 평가 유형에 사용자 지정 evaluator를 지원합니다. 이 프레임워크의 핵심은 rubric 체계입니다. 기준을 정의하면 LLM이 이를 일관되게 적용합니다.

**워크플로:**
1. 세션 검색 노트북에서 세션을 불러옵니다(또는 사용자 지정 세션 ID를 직접 제공합니다).
2. 각 세션의 trace를 가져와 평가 Case를 생성하고 evaluator를 실행합니다.
3. 결과를 EMF 형식으로 AgentCore에 기록합니다.
4. 요약 통계를 생성합니다.

**사전 요구 사항:** 먼저 세션 검색 노트북을 실행하거나 세션 ID 목록을 준비합니다.

## 전체 흐름에서의 위치

이 노트북은 **노트북 2(옵션 A)**입니다. 직접 정의한 사용자 지정 rubric을 사용해 세션을 평가합니다.

![노트북 워크플로](images/notebook_workflow.svg)

## 데이터 흐름

평가 파이프라인은 AgentCore Observability trace를 점수가 포함된 결과로 변환합니다.

![평가 파이프라인](images/evaluation_pipeline.svg)

## 설정

Strands Evals evaluator와 AgentCore Observability 연동을 위한 utility class 등 필요한 module을 가져옵니다. 구성은 `config.py`에서 불러옵니다.

In [ ]:
import logging
import sys
from datetime import datetime, timedelta, timezone
from typing import List

sys.path.insert(0, ".")

from config import (
    AWS_REGION,
    SOURCE_LOG_GROUP,
    EVAL_RESULTS_LOG_GROUP,
    LOOKBACK_HOURS,
    MAX_CASES_PER_SESSION,
    DISCOVERED_SESSIONS_PATH,
    RESULTS_JSON_PATH,
    EVALUATION_CONFIG_ID,
    setup_cloudwatch_environment,
)

from utils import (
    CloudWatchSessionMapper,
    ObservabilityClient,
    SessionDiscoveryResult,
    SessionInfo,
    send_evaluation_to_cloudwatch,
)

from strands_evals import Case, Experiment
from strands_evals.evaluators import OutputEvaluator, TrajectoryEvaluator
from strands_evals.types.trace import AgentInvocationSpan, ToolExecutionSpan

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

## 구성

CloudWatch metric에 사용할 evaluator 이름을 정의합니다. 이 이름은 AgentCore Observability 대시보드에 표시되며 `Custom.YourEvaluatorName` 규칙을 따라야 합니다. `EVALUATION_CONFIG_ID`는 `config.py`에서 불러옵니다.

In [ ]:
# CloudWatch metric에 사용할 사용자 지정 evaluator 이름입니다(사용 사례에 맞게 수정하세요).
OUTPUT_EVALUATOR_NAME = "Custom.OutputEvaluator"
TRAJECTORY_EVALUATOR_NAME = "Custom.TrajectoryEvaluator"

## CloudWatch 환경

평가 결과를 기록하는 데 필요한 환경 변수를 구성합니다. OTEL resource attribute에는 `config.py`의 `SERVICE_NAME`을 사용합니다.

In [ ]:
setup_cloudwatch_environment()

## 세션 불러오기

검색 노트북의 JSON 출력에서 세션을 불러옵니다. 특정 세션만 다시 평가하려면 `USE_JSON_FILE = False`로 설정하고 사용자 지정 세션 ID를 직접 제공할 수도 있습니다.

In [ ]:
# 사용자 지정 세션 ID를 직접 제공하려면 False로 설정합니다.
USE_JSON_FILE = True

if USE_JSON_FILE:
    discovery_result = SessionDiscoveryResult.load_from_json(DISCOVERED_SESSIONS_PATH)
    sessions_to_process = discovery_result.sessions
else:
    # 여기에 사용자 지정 세션 ID를 입력합니다.
    session_ids = [
        "your-session-id-here",
    ]
    sessions_to_process = [
        SessionInfo(
            session_id=sid,
            span_count=0,
            first_seen=datetime.now(timezone.utc),
            last_seen=datetime.now(timezone.utc),
            discovery_method="user_provided",
        )
        for sid in session_ids
    ]

print(f"Loaded {len(sessions_to_process)} sessions")

## Evaluator Rubric

rubric은 평가 기준을 정의합니다. evaluator는 rubric과 에이전트 출력을 판정자 역할의 LLM에 함께 전송하며, LLM은 설명과 함께 점수(0.0~1.0)를 반환합니다.

**효과적인 rubric 작성 방법:**
- 좋은 품질과 낮은 품질을 구분하는 조건을 구체적으로 작성합니다.
- 점수 기준점(1.0, 0.5, 0.0이 각각 무엇을 의미하는지)을 포함합니다.
- 에이전트의 domain과 관련된 측정 가능한 기준에 집중합니다.

아래 rubric을 사용자 지정합니다. 기본 rubric은 일반적인 응답 품질과 tool 사용 pattern을 평가합니다.

In [ ]:
output_rubric = """
Evaluate the agent's response based on:
1. Relevance: Does the response directly address the user's question?
2. Accuracy: Is the information factually correct?
3. Completeness: Does the response provide sufficient detail?

Score 0.0-1.0: 1.0=excellent, 0.5=adequate, 0.0=poor
"""

trajectory_rubric = """
Evaluate the agent's tool usage based on:
1. Tool Selection: Did the agent choose appropriate tools?
2. Efficiency: Were tools used without unnecessary calls?
3. Logical Sequence: Were tools used in a logical order?

Score 0.0-1.0: 1.0=optimal, 0.5=acceptable, 0.0=poor
"""

## 도우미 함수

다음 함수는 AgentCore Observability trace와 Strands Evals를 연결합니다.

- `task_fn(case)`: OutputEvaluator가 rubric을 기준으로 점수를 산정할 수 있도록 에이전트의 실제 응답을 반환합니다.

- `trajectory_task_fn(case)`: TrajectoryEvaluator가 tool 사용 pattern을 평가할 수 있도록 응답과 tool 순서를 모두 반환합니다.

- `create_cases_from_session(session)`: Strands Eval Session을 평가 Case로 변환합니다. AgentInvocationSpan에서 user prompt를, ToolExecutionSpan object에서 tool 이름을 추출하고, CloudWatch 연계를 위해 원본 trace_id를 유지합니다.

- `log_case_result_to_cloudwatch(case, ...)`: 원본 trace_id를 사용해 평가 결과를 AgentCore Observability로 전송합니다. 따라서 대시보드에서 원본 trace와 점수를 함께 확인할 수 있습니다.

In [ ]:
def task_fn(case: Case) -> str:
    """트레이스 메타데이터에서 실제 출력을 반환합니다."""
    return case.metadata.get("actual_output", "")


def trajectory_task_fn(case: Case):
    """트레이스 메타데이터에서 출력과 trajectory를 반환합니다."""
    return {
        "output": case.metadata.get("actual_output", ""),
        "trajectory": case.metadata.get("trajectory_for_eval", []),
    }


def log_case_result_to_cloudwatch(
    case: Case, evaluator_name: str, score: float, explanation: str, label: str = None
) -> bool:
    """원래 trace ID와 함께 평가 결과를 CloudWatch에 기록합니다."""
    trace_id = case.metadata.get("trace_id", "")
    if not trace_id:
        return False
    return send_evaluation_to_cloudwatch(
        trace_id=trace_id,
        session_id=case.session_id,
        evaluator_name=evaluator_name,
        score=score,
        explanation=explanation,
        label=label,
        config_id=EVALUATION_CONFIG_ID,
    )


def create_cases_from_session(session, session_id: str, max_cases: int = None) -> List[Case]:
    """Strands Eval Session에서 평가 케이스를 생성합니다."""
    cases = []
    for i, trace in enumerate(session.traces):
        if max_cases and len(cases) >= max_cases:
            break
        agent_span = None
        tool_names = []
        for span in trace.spans:
            if isinstance(span, AgentInvocationSpan):
                agent_span = span
            elif isinstance(span, ToolExecutionSpan):
                tool_names.append(span.tool_call.name)
        if agent_span:
            case = Case(
                name=f"trace_{i + 1}_{trace.trace_id[:8]}",
                input=agent_span.user_prompt or "",
                expected_output="",
                session_id=session_id,
                metadata={
                    "actual_output": agent_span.agent_response or "",
                    "actual_trajectory": tool_names,
                    "trace_id": trace.trace_id,
                    "tool_count": len(tool_names),
                },
            )
            cases.append(case)
    return cases

## 클라이언트 초기화

trace를 가져올 `ObservabilityClient`와 이를 변환할 `CloudWatchSessionMapper`를 생성합니다.

mapper는 원시 AgentCore Observability span을 구조화된 Strands Eval object로 변환합니다.
- trace_id별로 span을 그룹화하여 각 상호작용을 재구성합니다.
- tool 호출을 추출하고 해당 결과와 연결합니다.
- user prompt(첫 메시지)와 에이전트 응답(최종 출력)을 식별합니다.
- AgentInvocationSpan(전체 상호작용)과 ToolExecutionSpan(각 tool 사용)을 생성합니다.

In [ ]:
obs_client = ObservabilityClient(
    region_name=AWS_REGION,
    log_group=SOURCE_LOG_GROUP,
)
mapper = CloudWatchSessionMapper()

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=LOOKBACK_HOURS)
start_time_ms = int(start_time.timestamp() * 1000)
end_time_ms = int(end_time.timestamp() * 1000)

## 세션 처리

주요 평가 루프입니다. 각 세션에 대해 다음 작업을 수행합니다.
1. AgentCore Observability에서 span을 가져옵니다.
2. mapper를 사용해 span을 Strands Eval Session 형식으로 변환합니다.
3. 세션의 각 trace에서 평가 Case를 생성합니다.
4. 모든 case에 OutputEvaluator를 실행합니다.
5. tool을 사용한 case에 TrajectoryEvaluator를 실행합니다.
6. 대시보드에서 연계해 볼 수 있도록 원본 trace ID와 함께 모든 결과를 AgentCore Observability에 기록합니다.

각 세션의 진행 상황이 출력됩니다. 오류가 발생해도 루프를 중지하지 않고 오류를 포착해 기록합니다.

In [ ]:
all_session_results = []
total_cases_evaluated = 0
total_logs_sent = 0
all_tools_used = set()

for session_idx, session_info in enumerate(sessions_to_process):
    session_id = session_info.session_id
    print(f"[{session_idx + 1}/{len(sessions_to_process)}] {session_id}")

    try:
        trace_data = obs_client.get_session_data(
            session_id=session_id,
            start_time_ms=start_time_ms,
            end_time_ms=end_time_ms,
            include_runtime_logs=False,
        )

        if not trace_data.spans:
            all_session_results.append({"session_id": session_id, "status": "skipped", "reason": "no_spans"})
            continue

        session = trace_data.to_session(mapper)
        cases = create_cases_from_session(session, session_id, MAX_CASES_PER_SESSION)

        if not cases:
            all_session_results.append({"session_id": session_id, "status": "skipped", "reason": "no_cases"})
            continue

        for case in cases:
            for tool in case.metadata.get("actual_trajectory", []):
                all_tools_used.add(tool)

        # OutputEvaluator를 실행합니다.
        output_experiment = Experiment(cases=cases, evaluators=[OutputEvaluator(rubric=output_rubric)])
        output_results = output_experiment.run_evaluations(task_fn)
        output_report = output_results[0]

        output_logged = 0
        for i, case in enumerate(cases):
            if log_case_result_to_cloudwatch(
                case,
                OUTPUT_EVALUATOR_NAME,
                output_report.scores[i],
                output_report.reasons[i] if i < len(output_report.reasons) else "",
            ):
                output_logged += 1

        # TrajectoryEvaluator를 실행합니다.
        trajectory_cases = [c for c in cases if c.metadata.get("actual_trajectory")]
        trajectory_score = None
        trajectory_logged = 0

        if trajectory_cases:
            traj_eval_cases = [
                Case(
                    name=c.name,
                    input=c.input,
                    expected_output=c.expected_output,
                    session_id=c.session_id,
                    metadata={
                        **c.metadata,
                        "trajectory_for_eval": c.metadata.get("actual_trajectory", []),
                    },
                )
                for c in trajectory_cases
            ]
            trajectory_experiment = Experiment(
                cases=traj_eval_cases,
                evaluators=[
                    TrajectoryEvaluator(
                        rubric=trajectory_rubric,
                        trajectory_description={"available_tools": list(all_tools_used)},
                    )
                ],
            )
            trajectory_results = trajectory_experiment.run_evaluations(trajectory_task_fn)
            trajectory_report = trajectory_results[0]
            trajectory_score = trajectory_report.overall_score

            for i, case in enumerate(traj_eval_cases):
                if log_case_result_to_cloudwatch(
                    case,
                    TRAJECTORY_EVALUATOR_NAME,
                    trajectory_report.scores[i],
                    trajectory_report.reasons[i] if i < len(trajectory_report.reasons) else "",
                ):
                    trajectory_logged += 1

        all_session_results.append(
            {
                "session_id": session_id,
                "status": "completed",
                "case_count": len(cases),
                "output_score": output_report.overall_score,
                "trajectory_score": trajectory_score,
                "logs_sent": output_logged + trajectory_logged,
            }
        )
        total_cases_evaluated += len(cases)
        total_logs_sent += output_logged + trajectory_logged

    except Exception as e:
        all_session_results.append({"session_id": session_id, "status": "error", "error": str(e)})

print(
    f"\nCompleted: {len([r for r in all_session_results if r['status'] == 'completed'])} sessions, {total_cases_evaluated} cases, {total_logs_sent} logs sent"
)

## 요약

완료율, 평가한 전체 case 수, output 및 trajectory evaluator의 평균 점수 등 평가된 모든 세션의 통계를 집계합니다.

In [ ]:
completed = [r for r in all_session_results if r.get("status") == "completed"]
output_scores = [r["output_score"] for r in completed if r.get("output_score") is not None]
trajectory_scores = [r["trajectory_score"] for r in completed if r.get("trajectory_score") is not None]

print(f"Sessions: {len(completed)}/{len(all_session_results)} completed")
print(f"Cases evaluated: {total_cases_evaluated}")
print(f"CloudWatch logs sent: {total_logs_sent}")

if output_scores:
    print(
        f"Output score: avg={sum(output_scores) / len(output_scores):.2f}, min={min(output_scores):.2f}, max={max(output_scores):.2f}"
    )
if trajectory_scores:
    print(
        f"Trajectory score: avg={sum(trajectory_scores) / len(trajectory_scores):.2f}, min={min(trajectory_scores):.2f}, max={max(trajectory_scores):.2f}"
    )

## 세션별 결과

각 세션의 output 및 trajectory 점수가 포함된 개별 결과를 표시합니다. "skipped"로 표시된 세션에는 span 또는 유효한 case가 없으며, "error"로 표시된 세션은 처리 중 예외가 발생한 경우입니다.

In [ ]:
for i, r in enumerate(all_session_results):
    status = r.get("status", "unknown")
    if status == "completed":
        print(
            f"{i + 1}. {r['session_id'][:20]}... output={r.get('output_score', 0):.2f} traj={r.get('trajectory_score') or '-'}"
        )
    else:
        print(f"{i + 1}. {r['session_id'][:20]}... {status}")

## 결과 내보내기

추가 분석이나 보고에 사용할 수 있도록 평가 결과를 JSON으로 저장합니다. 내보낸 데이터에는 구성, 요약 통계, 세션별 결과가 포함됩니다.

In [ ]:
import json

export_data = {
    "evaluation_time": datetime.now(timezone.utc).isoformat(),
    "config": {
        "source_log_group": SOURCE_LOG_GROUP,
        "eval_results_log_group": EVAL_RESULTS_LOG_GROUP,
        "output_evaluator": OUTPUT_EVALUATOR_NAME,
        "trajectory_evaluator": TRAJECTORY_EVALUATOR_NAME,
    },
    "summary": {
        "total_sessions": len(all_session_results),
        "completed_sessions": len(completed),
        "total_cases": total_cases_evaluated,
        "avg_output_score": sum(output_scores) / len(output_scores) if output_scores else None,
        "avg_trajectory_score": sum(trajectory_scores) / len(trajectory_scores) if trajectory_scores else None,
    },
    "session_results": all_session_results,
}

with open(RESULTS_JSON_PATH, "w") as f:
    json.dump(export_data, f, indent=2)

print(f"Exported to {RESULTS_JSON_PATH}")